# Document Graph Demo

This notebook shows how to use `genai-graph` to build a **generic Document+Chunk knowledge graph** from a directory of text files.

## What you'll do
1. Configure a directory and graph output path
2. Run the `DocumentDirectoryFactory` to ingest files → Document nodes
3. (Optional) Add Chunk nodes via semantic chunking
4. Query the resulting graph with Cypher
5. Visualize the graph inline

**No LLM needed** for the basic graph — just files and the graph engine.

> To extend with entity extraction, see `DocumentDirectoryFactory` docstring for subclassing instructions.

## 1. Configuration

Edit the paths below to point at your documents directory and desired output location.

In [1]:
import tempfile
from pathlib import Path

# ── Edit these paths ──────────────────────────────────────────────────────
# Directory containing documents to ingest (markdown / text files)
DOCUMENTS_DIR = Path("docs")  # genai-graph docs as a sample corpus

# Where to store the Kuzu graph database
DB_PATH = Path(tempfile.mkdtemp()) / "document_graph.db"
# ─────────────────────────────────────────────────────────────────────────

DOCUMENTS_DIR = DOCUMENTS_DIR.resolve()
print(f"Documents : {DOCUMENTS_DIR}")
print(f"Graph DB  : {DB_PATH}")
print(f"Files found: {len(list(DOCUMENTS_DIR.rglob('*.md')))} markdown files")

Documents : /home/tcl/prj/genai-graph/docs
Graph DB  : /tmp/tmpyt6pr5rb/document_graph.db
Files found: 7 markdown files


## 2. Build the Document Graph

The `DocumentDirectoryFactory` scans the directory and creates:
- One **Document** node per file (path, filename, size, hash, mime-type)
- **Chunk** nodes (semantic chunks using *chonkie*) linked by `CONTAINS` and `NEXT`

In [2]:
from genai_graph.kg.backend import KuzuBackend
from genai_graph.kg.factories.document_factory import DocumentDirectoryFactory
from genai_graph.kg.ingest.extract import create_schema
from genai_graph.kg.ingest.merge import merge_nodes_batch, merge_relationships_batch

# 1. Open (or create) the graph database
backend = KuzuBackend()
backend.connect(str(DB_PATH))
print(f"✅ Opened graph DB at {DB_PATH}")

# 2. Initialise the factory
factory = DocumentDirectoryFactory(
    data_root=str(DOCUMENTS_DIR),
    include=["*.md", "*.txt", "*.rst"],
    chunk_size=512,
    overlap=50,
)

schema = factory.build_schema()
print(f"Schema nodes     : {[n.node_class.__name__ for n in schema.nodes]}")
print(f"Schema relations : {[r.name for r in schema.relations]}")

✅ Opened graph DB at /tmp/tmpyt6pr5rb/document_graph.db
Schema nodes     : ['Document', 'Chunk']
Schema relations : ['CONTAINS', 'NEXT']


/home/tcl/prj/genai-graph/genai_graph/kg/schema/core.py:388: UserWarning: Graph schema validation: No field paths found for Chunk in the root model structure; this node may be orphaned.
  self._validate_coherence(context=None)  # Context not available during model validation


In [3]:
# 3. Create table schema in Kuzu
create_schema(backend, schema.nodes, schema.relations)
print("✅ Schema created")

2026-06-27 14:30:54.981 | DEBUG    | genai_graph.kg.ingest.extract:create_schema:474 - Creating node table: CREATE NODE TABLE IF NOT EXISTS Document(name STRING, _original_name STRING, _created_at STRING, _updated_at STRING, path STRING, filename STRING, file_size INT64, mime_type STRING, modified_at STRING, content_hash STRING, access_level STRING, allowed_roles STRING[], allowed_users STRING[], PRIMARY KEY(path))
2026-06-27 14:30:55.390 | DEBUG    | genai_graph.kg.ingest.extract:create_schema:474 - Creating node table: CREATE NODE TABLE IF NOT EXISTS Chunk(name STRING, _original_name STRING, _created_at STRING, _updated_at STRING, chunk_id STRING, document_path STRING, text STRING, chunk_index INT64, start_offset INT64, end_offset INT64, token_count INT64, embedding FLOAT[], embedding_embedding FLOAT[1536], PRIMARY KEY(chunk_id))
2026-06-27 14:30:55.394 | DEBUG    | genai_graph.kg.ingest.extract:create_schema:506 - Creating relationship table: CREATE REL TABLE IF NOT EXISTS CONTAINS(

✅ Schema created


In [4]:
from genai_graph.kg.nodes.document import Chunk, Document

all_doc_nodes = []
all_chunk_nodes = []
all_contains_rels = []  # Document → Chunk
all_next_rels = []  # Chunk → Chunk

keys = factory.get_keys()
print(f"Ingesting {len(keys)} file(s) …")

for file_path in keys:
    doc = factory.get_struct_data_by_key(file_path)
    if doc is None:
        continue
    all_doc_nodes.append(doc)

    chunks = factory.build_document_chunks(file_path)
    for i, chunk in enumerate(chunks):
        all_chunk_nodes.append(chunk)
        all_contains_rels.append((doc.path, chunk.chunk_id))  # Document → Chunk
        if i > 0:
            all_next_rels.append((chunks[i - 1].chunk_id, chunk.chunk_id))  # Chunk → Chunk

print(f"  Documents : {len(all_doc_nodes)}")
print(f"  Chunks    : {len(all_chunk_nodes)}")
print(f"  CONTAINS  : {len(all_contains_rels)}")
print(f"  NEXT      : {len(all_next_rels)}")

2026-06-27 14:30:55.410 | INFO     | genai_graph.kg.factories.document_factory:_get_files:210 - DocumentDirectoryFactory: discovered 7 files under /home/tcl/prj/genai-graph/docs


Ingesting 7 file(s) …
  Documents : 7
  Chunks    : 186
  CONTAINS  : 186
  NEXT      : 179


In [5]:
from datetime import datetime

from genai_graph.kg.ingest.extract import RelationshipRecord, import_neo4j_data
from genai_graph.kg.ingest.merge import NodeDataCollection

now = datetime.utcnow().isoformat() + "Z"

nodes_data = NodeDataCollection()
for doc in all_doc_nodes:
    d = doc.model_dump()
    d["name"] = doc.filename
    d["_created_at"] = now
    d["_updated_at"] = now
    nodes_data.add("Document", d)

for chunk in all_chunk_nodes:
    c = chunk.model_dump()
    c.pop("embedding", None)  # skip None embeddings (avoid FLOAT[] type conflict)
    c["name"] = chunk.chunk_id
    c["_created_at"] = now
    c["_updated_at"] = now
    nodes_data.add("Chunk", c)

relationships = []
for doc_path, chunk_id in all_contains_rels:
    relationships.append(RelationshipRecord("Document", doc_path, "Chunk", chunk_id, "CONTAINS", {}))
for from_cid, to_cid in all_next_rels:
    relationships.append(RelationshipRecord("Chunk", from_cid, "Chunk", to_cid, "NEXT", {}))

import_neo4j_data(backend, nodes_data, relationships, key_fields={"Document": "path", "Chunk": "chunk_id"})
print(f"✅ Graph written: {nodes_data.total_count()} nodes, {len(relationships)} relationships")

/tmp/ipykernel_95804/2611970185.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow().isoformat() + "Z"
2026-06-27 14:30:55.946 | INFO     | genai_graph.kg.ingest.extract:import_neo4j_data:890 - Importing 193 nodes and 365 relationships
2026-06-27 14:30:55.948 | DEBUG    | genai_graph.kg.ingest.extract:_create_dynamic_schema_for_nodes:1075 - Creating dynamic node table: Document
2026-06-27 14:30:55.950 | DEBUG    | genai_graph.kg.ingest.extract:_create_dynamic_schema_for_nodes:1075 - Creating dynamic node table: Chunk
2026-06-27 14:30:55.953 | DEBUG    | genai_graph.kg.ingest.extract:_create_dynamic_schema_for_nodes:1138 - Creating dynamic rel table: CONTAINS
2026-06-27 14:30:55.954 | DEBUG    | genai_graph.kg.ingest.extract:_create_dynamic_schema_for_nodes:1138 - Creating dynamic rel table: NEXT
2026-06-27 

✅ Graph written: 193 nodes, 365 relationships


## 3. Query the Graph

Use standard Cypher to explore the graph.

In [6]:
from rich.console import Console
from rich.table import Table

console = Console()


def run_query(cypher: str, title: str = "Results") -> None:
    """Execute a Cypher query and display as a Rich table."""
    try:
        df = backend.execute_get_as_df(cypher, union=True)
        if df.empty:
            console.print(f"[yellow]{title}: no results[/yellow]")
            return
        table = Table(title=f"{title} ({len(df)} rows)")
        for col in df.columns:
            table.add_column(str(col), style="cyan")
        for _, row in df.head(20).iterrows():
            table.add_row(*[str(v) for v in row])
        console.print(table)
        if len(df) > 20:
            console.print(f"[dim]… {len(df) - 20} more rows[/dim]")
    except Exception as exc:
        console.print(f"[red]Query error: {exc}[/red]")

In [7]:
# Node counts by type
run_query("MATCH (d:Document) RETURN d.filename, d.file_size, d.mime_type ORDER BY d.filename", "Documents")

                      Documents (7 rows)                       
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ d.filename                    ┃ d.file_size ┃ d.mime_type   ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ baml_extraction_guide.md      │ 24376       │ text/markdown │
│ cache_management.md           │ 2922        │ text/markdown │
│ graph_construction.md         │ 21676       │ text/markdown │
│ kg_explorer.md                │ 5791        │ text/markdown │
│ prefect_dag_pipeline.md       │ 12636       │ text/markdown │
│ primary_key_implementation.md │ 6732        │ text/markdown │
│ workflows.md                  │ 12295       │ text/markdown │
└───────────────────────────────┴─────────────┴───────────────┘

In [8]:
# Chunks per document
run_query(
    """
    MATCH (d:Document)-[:CONTAINS]->(c:Chunk)
    RETURN d.filename AS document, count(c) AS chunks
    ORDER BY chunks DESC
    """,
    "Chunks per Document",
)

       Chunks per Document (7 rows)       
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ document                      ┃ chunks ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ baml_extraction_guide.md      │ 53     │
│ graph_construction.md         │ 45     │
│ prefect_dag_pipeline.md       │ 27     │
│ workflows.md                  │ 26     │
│ primary_key_implementation.md │ 15     │
│ kg_explorer.md                │ 13     │
│ cache_management.md           │ 7      │
└───────────────────────────────┴────────┘

In [9]:
# Show first chunk of each document
run_query(
    """
    MATCH (d:Document)-[:CONTAINS]->(c:Chunk)
    WHERE c.chunk_index = 0
    RETURN d.filename AS document, c.text AS first_chunk
    ORDER BY d.filename
    """,
    "First Chunk per Document",
)

                                         First Chunk per Document (7 rows)                                         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ document                      ┃ first_chunk                                                                     ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ baml_extraction_guide.md      │ # BAML Extraction and Graph Factory Creation Guide                              │
│                               │                                                                                 │
│                               │ This guide covers genai-graph–specific patterns for creating graph factories    │
│                               │ that consume                                                                    │
│                               │ BAML-extracted data. For BAML setup fundamentals (writing `.baml` files,        │
│                               │ generating Python                                                               │
│                               │ types, configuring the LLM client, and running `cli baml extract`), see the     │
│                               │ (../../genai-tk/docs/baml.md).                                                  │
│                               │                                                                                 │
│                               │ ## Quick Reference                                                              │
│                               │                                                                                 │
│                               │ | Task | Files to Modify | Command |                                            │
│                               │ |------|----------------|---------|                                             │
│                               │ | Add/modify extracted fields | `genai_graph/ekg/                               │
│ cache_management.md           │ # Knowledge Graph Parquet Cache Management                                      │
│                               │                                                                                 │
│                               │ The KG system caches imported graph data in Parquet format so that dependent KG │
│                               │ configurations don't re-process unchanged sources on every build. This document │
│                               │ explains                                                                        │
│                               │ when caches become invalid and how to clear them.                               │
│                               │                                                                                 │
│                               │ ## Cache Location                                                               │
│                               │                                                                                 │
│                               │ ```                                                                             │
│                               │ ~/kg_outputs/{kg_name}/parquet/   ← one directory per KG name                   │
│                               │ ```                                                                             │
│                               │                                                                                 │
│                               │ A `manifest.json` inside each directory tracks content fingerprints. If         │
│                               │ fingerprints match,                                                             │
│                               │ the import phase is skipped.                                                    │
│                               │                       

## 4. Visualize the Graph

Generate an interactive HTML graph and display it inline.

In [10]:
from IPython.display import HTML, display

from genai_graph.kg.export.html import generate_html

# Fetch a compact subgraph for visualization (documents + first chunk of each)
cypher = """
MATCH (d:Document)-[:CONTAINS]->(c:Chunk)
WHERE c.chunk_index = 0
RETURN d, c
LIMIT 50
"""

try:
    html_content = generate_html(
        connection=backend,
        destination_file_path="/tmp/document_graph.html",
        query=cypher,
    )
    display(HTML(html_content))
except Exception as exc:
    # Fallback: show schema-based visualization
    print(f"Graph HTML not available: {exc}")
    print("Use 'cli kg view' after running the document_graph workflow for a full visualization.")

Error in _fetch_graph_data: 'n'


## 5. Schema Visualization

Display the schema (node types and relationships) as an interactive D3 diagram.

In [12]:
from IPython.display import HTML, display

schema = factory.build_schema()

try:
    from genai_graph.kg.schema.schema_html import render_schema_html

    schema_html = render_schema_html(schema, title="Document+Chunk Schema")
    display(HTML(f'<iframe srcdoc="{schema_html.replace(chr(34), chr(39))}" width="100%" height="500"></iframe>'))
except Exception as exc:
    print(f"Schema visualization not available: {exc}")
    # Fallback: text summary
    print("\nNode types:")
    for node in schema.nodes:
        print(f"  - {node.node_class.__name__}")
    print("\nRelationships:")
    for rel in schema.relations:
        print(f"  - {rel.from_node.node_class.__name__} -[{rel.name}]-> {rel.to_node.node_class.__name__}")

Schema visualization not available: cannot import name 'render_schema_html' from 'genai_graph.kg.schema.schema_html' (/home/tcl/prj/genai-graph/genai_graph/kg/schema/schema_html.py)

Node types:
  - Document
  - Chunk

Relationships:
  - Document -[CONTAINS]-> Chunk
  - Chunk -[NEXT]-> Chunk


## Alternative: Use the CLI Workflow

Instead of the manual steps above, you can use the bundled `document_graph` workflow:

```bash
# Dry-run to see the plan
uv run cli workflow run document_graph --dry-run

# Run with default settings (scans ${paths.data_root}/documents)
uv run cli workflow run document_graph

# Run with custom directory
uv run cli workflow run document_graph --set data_dir=/path/to/docs

# Open the HTML visualization
uv run cli kg view
```

The workflow handles schema creation, ingestion, and HTML export automatically.

## Next Steps: Entity Extraction

To add LLM-based entity extraction on top of the Document+Chunk graph:

1. Define a BAML schema with your domain entities
2. Subclass `DocumentDirectoryFactory` and override `build_schema()` to add your entity nodes
3. Override `get_struct_data_by_key()` to call BAML extraction per document

See `genai_graph/kg/factories/document_factory.py` docstring and the `ekg-atos` project
for a full example of this pattern.